### Prepare config file

`Inosice` similar substructures has been downloaded from Pubchem (https://pubchem.ncbi.nlm.nih.gov/#query=CID135398641+structure&tab=substructure&cid=135398641). Now we are going to convert it to csv file to run `DORAnet` for all the possible substructures. 

In [9]:
from pathlib import Path

import pandas as pd
from rdkit import Chem


def sdfToDataFrame(sdfPath: str) -> pd.DataFrame:
    """
    Load an SDF file into a pandas DataFrame.
    - One row per molecule
    - Columns include: SDF properties + SMILES + InChIKey
    """
    supplier = Chem.SDMolSupplier(sdfPath, removeHs=False)
    rows = []

    for idx, mol in enumerate(supplier):
        if mol is None:
            continue

        rowDict = {"recordIndex": idx}

        # Add all SDF properties as columns
        for propName in mol.GetPropNames():
            rowDict[propName] = mol.GetProp(propName)

        # Add a few useful computed identifiers
        rowDict["smiles"] = Chem.MolToSmiles(mol, isomericSmiles=True)
        rowDict["inchiKey"] = Chem.inchi.MolToInchiKey(mol)

        rows.append(rowDict)

    DF = pd.DataFrame(rows)
    return DF


def canonicalizeSmiles(smilesStr: str) -> str | None:
    """
    Canonicalize a SMILES string with RDKit.
    Returns None if parsing fails or input is missing.
    """
    if smilesStr is None:
        return None

    smilesStr = str(smilesStr).strip()
    if smilesStr == "" or smilesStr.lower() == "nan":
        return None

    mol = Chem.MolFromSmiles(smilesStr)
    if mol is None:
        return None

    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)


def ensureSmilesFirst(DF: pd.DataFrame, targetSmiles: str) -> pd.DataFrame:
    """
    Ensure `targetSmiles` appears as the first row in DF (based on exact SMILES string match),
    inserting it if it is not already present.
    """
    if "smiles" not in DF.columns:
        raise KeyError("Expected a 'smiles' column, but it was not found in the DataFrame.")

    targetSmiles = targetSmiles.strip()

    # Exact-match check (after stripping whitespace)
    existingSmilesSet = set(DF["smiles"].astype(str).str.strip().tolist())
    if targetSmiles in existingSmilesSet:
        # If present but not first, move it to top
        mask = DF["smiles"].astype(str).str.strip().eq(targetSmiles)
        DFTop = DF.loc[mask]
        DFRest = DF.loc[~mask]
        return pd.concat([DFTop, DFRest], ignore_index=True)

    # If not present, create a new row and prepend it
    mol = Chem.MolFromSmiles(targetSmiles)
    if mol is None:
        raise ValueError("The target SMILES could not be parsed by RDKit.")

    newRow = {
        "recordIndex": -1,  # sentinel for inserted row
        "smiles": targetSmiles,
        "inchiKey": Chem.inchi.MolToInchiKey(mol),
    }

    # Ensure any additional columns exist in new row (fill with None)
    for colName in DF.columns:
        if colName not in newRow:
            newRow[colName] = None

    DFNew = pd.DataFrame([newRow], columns=DF.columns)
    return pd.concat([DFNew, DF], ignore_index=True)


def saveDfAsCsvSameName(sdfPath: str, DF: pd.DataFrame) -> str:
    sdfFilePath = Path(sdfPath)
    csvFilePath = sdfFilePath.with_suffix(".csv")
    DF.to_csv(csvFilePath, index=False, encoding="utf-8")
    return str(csvFilePath)


if __name__ == "__main__":
    sdfPath = "Inosine_PubChem_substructures.sdf"

    DF = sdfToDataFrame(sdfPath)

    # ---- Ensure this SMILES is first (insert if missing) ----
    targetSmiles = "C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H](O3)CO)O)O"  # Inosine SMILES
    DF = ensureSmilesFirst(DF, targetSmiles)

    # ---- Canonicalize SMILES into a new column ----
    DF["Canonical_SMILES"] = DF["smiles"].apply(canonicalizeSmiles)

    # ---- Save CSV ----
    csvPath = saveDfAsCsvSameName(sdfPath, DF)
    print(f"\nSaved CSV to: {csvPath}")


[11:41:03] WARNING: not removing hydrogen atom without neighbors
[11:41:03] WARNING: not removing hydrogen atom without neighbors
[11:41:03] WARNING: not removing hydrogen atom without neighbors
[11:41:03] WARNING: not removing hydrogen atom without neighbors
[11:41:03] WARNING: not removing hydrogen atom without neighbors
[11:41:03] WARNING: not removing hydrogen atom without neighbors
[11:41:03] WARNING: not removing hydrogen atom without neighbors
[11:41:03] WARNING: not removing hydrogen atom without neighbors
[11:41:03] WARNING: not removing hydrogen atom without neighbors
[11:41:03] WARNING: not removing hydrogen atom without neighbors
[11:41:03] WARNING: not removing hydrogen atom without neighbors
[11:41:03] WARNING: not removing hydrogen atom without neighbors



Saved CSV to: Inosine_PubChem_substructures.csv


[11:41:03] WARNING: not removing hydrogen atom without neighbors


In [10]:
DF

,recordIndex,PUBCHEM_COMPOUND_CID,PUBCHEM_COMPOUND_CANONICALIZED,PUBCHEM_CACTVS_COMPLEXITY,PUBCHEM_CACTVS_HBOND_ACCEPTOR,PUBCHEM_CACTVS_HBOND_DONOR,PUBCHEM_CACTVS_ROTATABLE_BOND,PUBCHEM_CACTVS_SUBSKEYS,PUBCHEM_IUPAC_OPENEYE_NAME,PUBCHEM_IUPAC_CAS_NAME,PUBCHEM_IUPAC_NAME_MARKUP,PUBCHEM_IUPAC_NAME,PUBCHEM_IUPAC_SYSTEMATIC_NAME,PUBCHEM_IUPAC_TRADITIONAL_NAME,PUBCHEM_IUPAC_INCHI,PUBCHEM_IUPAC_INCHIKEY,PUBCHEM_XLOGP3,PUBCHEM_EXACT_MASS,PUBCHEM_MOLECULAR_FORMULA,PUBCHEM_MOLECULAR_WEIGHT,PUBCHEM_CACTVS_TPSA,PUBCHEM_MONOISOTOPIC_WEIGHT,PUBCHEM_TOTAL_CHARGE,PUBCHEM_HEAVY_ATOM_COUNT,PUBCHEM_ATOM_DEF_STEREO_COUNT,PUBCHEM_ATOM_UDEF_STEREO_COUNT,PUBCHEM_BOND_DEF_STEREO_COUNT,PUBCHEM_BOND_UDEF_STEREO_COUNT,PUBCHEM_ISOTOPIC_ATOM_COUNT,PUBCHEM_COMPONENT_COUNT,PUBCHEM_CACTVS_TAUTO_COUNT,PUBCHEM_COORDINATE_TYPE,PUBCHEM_BONDANNOTATIONS,smiles,inchiKey,PUBCHEM_XLOGP3_AA,PUBCHEM_NONSTANDARDBOND,Canonical_SMILES
0,-1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H](O3)CO)O)O,UGQMRVRMYYASKQ-KQYNXXCUSA-N,None,None,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O
1,0,135398641,1,405,7,4,2,AAADccBzuAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAEABgAAAHgAQ...,"9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydroxymethyl)tetrahyd...","9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydroxymethyl)-2-oxola...","9-[(2<I>R</I>,3<I>R</I>,4<I>S</I>,5<I>R</I>)-3,4-dihydro...","9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydroxymethyl)oxolan-2...","9-[(2R,3R,4S,5R)-5-(hydroxymethyl)-3,4-bis(oxidanyl)oxol...","9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-methylol-tetrahydrofura...",InChI=1S/C10H12N4O5/c15-1-4-6(16)7(17)10(19-4)14-3-13-5-...,UGQMRVRMYYASKQ-KQYNXXCUSA-N,-1.3,268.08076950,C10H12N4O5,268.23,129,268.08076950,0,19,4,0,0,0,0,1,-1,1\n5\n255,13 14 6\n15 17 8\n17 18 8\n10 2 5\n12 3 5\n11 ...,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])nc3c(=O)n(...,UGQMRVRMYYASKQ-KQYNXXCUSA-N,NaN,NaN,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O
2,1,135449284,1,658,22,13,14,AAADcfB//gAAAAAAAAAAAAAAAAAAAWJAAAAwYMEAAAAAAEAB1AAAHgAQ...,"tris(4-acetamidobenzoic acid);9-[(2R,3R,4S,5R)-3,4-dihyd...","tris(4-acetamidobenzoic acid);9-[(2R,3R,4S,5R)-3,4-dihyd...","tris(4-acetamidobenzoic acid);9-[(2<I>R</I>,3<I>R</I>,4<...","tris(4-acetamidobenzoic acid);9-[(2R,3R,4S,5R)-3,4-dihyd...",tris(4-acetamidobenzoic acid);tris(1-(dimethylamino)prop...,"tris(4-acetamidobenzoic acid);9-[(2R,3R,4S,5R)-3,4-dihyd...",InChI=1S/C10H12N4O5.3C9H9NO3.3C5H13NO/c15-1-4-6(16)7(17)...,YLDCUKJMEKGGFI-QCSRICIXSA-N,NaN,1114.55464106,C52H78N10O17,1115.2,399,1114.55464106,0,79,4,3,0,0,0,7,-1,1\n5\n255,30 18 6\n18 33 8\n18 34 8\n19 34 8\n19 35 8\n2...,[H]OC(=O)c1c([H])c([H])c(N([H])C(=O)C([H])([H])[H])c([H]...,YLDCUKJMEKGGFI-QCSRICIXSA-N,NaN,NaN,CC(=O)Nc1ccc(C(=O)O)cc1.CC(=O)Nc1ccc(C(=O)O)cc1.CC(=O)Nc...
3,2,135402037,1,405,7,4,2,AAADccBzuAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAEABgAAAHgAQ...,"9-[3,4-dihydroxy-5-(hydroxymethyl)tetrahydrofuran-2-yl]-...","9-[3,4-dihydroxy-5-(hydroxymethyl)-2-oxolanyl]-1H-purin-...","9-[3,4-dihydroxy-5-(hydroxymethyl)oxolan-2-yl]-1<I>H</I>...","9-[3,4-dihydroxy-5-(hydroxymethyl)oxolan-2-yl]-1H-purin-...","9-[5-(hydroxymethyl)-3,4-bis(oxidanyl)oxolan-2-yl]-1H-pu...","9-(3,4-dihydroxy-5-methylol-tetrahydrofuran-2-yl)hypoxan...",InChI=1S/C10H12N4O5/c15-1-4-6(16)7(17)10(19-4)14-3-13-5-...,UGQMRVRMYYASKQ-UHFFFAOYSA-N,-1.3,268.08076950,C10H12N4O5,268.23,129,268.08076950,0,19,0,4,0,0,0,1,-1,1\n5\n255,10 20 3\n11 21 3\n12 22 3\n13 23 3\n15 17 8\n1...,[H]OC([H])([H])C1([H])OC([H])(n2c([H])nc3c(=O)n([H])c([H...,UGQMRVRMYYASKQ-UHFFFAOYSA-N,NaN,NaN,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O
4,3,135992491,1,405,7,4,2,AAADccBzuAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAEABgAAAHgAQ...,"9-[(3S,4S,5R)-3,4-dihydroxy-5-(hydroxymethyl)tetrahydrof...","9-[(3S,4S,5R)-3,4-dihydroxy-5-(hydroxymethyl)-2-oxolanyl...","9-[(3<I>S</I>,4<I>S</I>,5<I>R</I>)-3,4-dihydroxy-5-(hydr...","9-[(3S,4S,5R)-3,4-dihydroxy-5-(hydr

### Generate max_atoms config from a target SMILES

In [12]:
from pathlib import Path
from collections import Counter

import numpy as np
from rdkit import Chem


def generateMaxAtomsConfig(targetSmiles: str, increasePercent: float = 0.5) -> dict:
    """
    Generate max_atoms config from a target SMILES.
    Increases each atom count by the specified percentage (default 50%).
    Returns a dictionary with atom limits.
    """
    mol = Chem.MolFromSmiles(targetSmiles)
    if mol is None:
        raise ValueError(f"Could not parse SMILES: {targetSmiles}")

    atomCounter = Counter(atom.GetSymbol() for atom in mol.GetAtoms())

    atomsOfInterest = ['C', 'N', 'O', 'S']

    maxAtoms = {}
    print(f"\nTarget molecule SMILES: {targetSmiles}")
    print(f"\nAtom counts in target molecule:")
    for atom in atomsOfInterest:
        count = atomCounter.get(atom, 0)
        suggested = int(np.ceil(count * (1 + increasePercent)))
        suggested = max(suggested, 1)
        maxAtoms[atom] = suggested
        print(f"  {atom}: {count}")

    print(f"\nGenerated max_atoms config ({int(increasePercent * 100)}% increase):")
    print(f"max_atoms:")
    for atom in atomsOfInterest:
        names = {'C': 'Carbon', 'N': 'Nitrogen', 'O': 'Oxygen', 'S': 'Sulfur'}
        print(f"  {atom}: {maxAtoms[atom]}   # {names[atom]}")

    return maxAtoms


if __name__ == "__main__":
    # ---- User supplies only this ----
    targetSmiles = "C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H](O3)CO)O)O"

    maxAtomsConfig = generateMaxAtomsConfig(targetSmiles)


Target molecule SMILES: C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H](O3)CO)O)O

Atom counts in target molecule:
  C: 10
  N: 4
  O: 5
  S: 0

Generated max_atoms config (50% increase):
max_atoms:
  C: 15   # Carbon
  N: 6   # Nitrogen
  O: 8   # Oxygen
  S: 1   # Sulfur
